# Анализ PT

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def plot_metric(series, title: str, ylabel: str, visible_start: int = 1):
    plt.figure(figsize=(13, 9))
    plt.plot(series, color="black")
    plt.xlabel("Epoch")
    plt.ylabel(ylabel)

    visible_data = series.iloc[visible_start:]
    y_min, y_max = visible_data.min(), visible_data.max()
    padding = (y_max - y_min) * 0.05 if y_max != y_min else 1e-3
    plt.xlim(visible_start, len(series))
    plt.ylim(y_min - padding, y_max + padding)
    plt.title(title)
    plt.tight_layout()
    plt.show()


def perform_log_log_analysis(series, metric_name: str, visible_start: int = 1):
    y_data = series.iloc[visible_start:]
    x_data = np.arange(visible_start, visible_start + len(y_data))

    if (y_data <= 0).any():
        print(
            f"Warning: {metric_name} has non-positive values -- log transform may not work correctly."
        )

    log_x = np.log(x_data)
    log_y = np.log(y_data)

    coeffs = np.polyfit(log_x, log_y, 1)
    alpha = coeffs[0]
    log_C = coeffs[1]
    C = np.exp(log_C)

    plt.figure(figsize=(13, 9))
    plt.loglog(x_data, y_data, "o", label="Data", color="black", markersize=2)

    x_fit = np.linspace(x_data.min(), x_data.max(), 200)
    y_fit = C * x_fit**alpha
    plt.loglog(x_fit, y_fit, label=f"Fit: y = {C:.3e} * x^{alpha:.3f}", color="red")

    plt.xlabel("Epoch (log scale)")
    plt.ylabel(f"{metric_name} (log scale)")
    plt.legend()
    plt.tight_layout()
    plt.show()

    print(f"Scaling law for {metric_name}: {metric_name} ~ {C:.3e} * epoch^{alpha:.3f}")
    return alpha, C

In [ ]:
FILE_NAME = ""

In [ ]:
avg_info = pd.read_csv(
    FILE_NAME,
    comment="#",
    header=None,
    names=["epoch", "mean_diff_mae", "mean_abs_mae", "grad_norm", "lr"],
    dtype={
        "epoch": np.int32,
        "mean_diff_mae": np.float64,
        "mean_abs_mae": np.float64,
        "grad_norm": np.float64,
        "lr": np.float64,
    },
)

In [ ]:
display(avg_info.head())
display(avg_info.tail())
display(avg_info.describe())

In [ ]:
VISIBLE_START = 1

In [ ]:
plot_metric(
    series=avg_info.mean_abs_mae,
    title="Mean Absolute MAE",
    ylabel="Mean Abs MAE",
    visible_start=VISIBLE_START,
)

In [ ]:
alpha_abs, C_abs = perform_log_log_analysis(
    series=avg_info.mean_abs_mae,
    metric_name="Mean Abs MAE",
    visible_start=VISIBLE_START,
)

In [ ]:
plot_metric(
    series=avg_info.mean_diff_mae,
    title="Mean Diff MAE",
    ylabel="Mean Diff MAE",
    visible_start=VISIBLE_START,
)

In [ ]:
alpha_diff, C_diff = perform_log_log_analysis(
    series=avg_info.mean_diff_mae,
    metric_name="Mean Diff MAE",
    visible_start=VISIBLE_START,
)

In [ ]:
plot_metric(
    series=avg_info.grad_norm,
    title="Gradient Norm",
    ylabel="|∇|",
    visible_start=VISIBLE_START,
)

In [ ]:
alpha_grad, C_grad = perform_log_log_analysis(
    series=avg_info.grad_norm, metric_name="Gradient Norm", visible_start=VISIBLE_START
)